# Notebook 1: Data Cleaning and Preprocessing

This notebook loads the raw Facebook event data, filters and cleans it, assigns ideology scores to users and events, and produces eight cleaned datasets (small/medium events × four annual periods) ready for graph construction.

**Outputs** saved to `../data/processed/`:
- `events_{size}_{period}.pkl` — 8 event DataFrames
- `users_{size}_{period}.pkl` — 8 user DataFrames

**Pipeline**
1. Load raw events and political user metadata
2. Filter events (duplicates, cancellations, coordinates, date range)
3. Geographic filter — Copenhagen and Frederiksberg municipalities
4. Build user dataset and merge with political metadata
5. Remove bot accounts (top 0.01% by events attended)
6. Iterative cleaning — remove events with no political attendees
7. Assign ideology scores to users and events
8. Split by event size (small ≤ 30, medium 31–100) and time period (annual, June–June)
9. Per-period iterative cleaning
10. Save outputs

In [ ]:
import ast
import warnings
from collections import Counter
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

RAW_DIR  = Path('../raw-data')
DATA_DIR = Path('../data/processed')
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Raw Data

In [ ]:
events = pd.read_csv(RAW_DIR / 'full_data_mod_pickle.csv')
political_users = pd.read_csv(RAW_DIR / 'uid_metadata_segmented.csv')

# Parse attending lists from CSV strings to Python sets immediately
# so all downstream code works with native Python collections.
events['attending'] = events['attending'].apply(
    lambda x: set(ast.literal_eval(x)) if isinstance(x, str) else set(x)
)

# Preserve original attendee information before any filtering
events['OG_n_attending'] = events['n_attending']
events['OG_attending']   = events['attending'].apply(list)

print(f"Events loaded:          {len(events):,}")
print(f"Political users loaded: {len(political_users):,}")

## 2. Filter Events

In [ ]:
print(f"Starting:                    {len(events):>8,}")

events = events.drop_duplicates(subset='id')
print(f"After dropping duplicates:   {len(events):>8,}")

events = events[~events['canceled']]
print(f"After removing canceled:     {len(events):>8,}")

events = events[events['n_attending'] > 1]
print(f"After requiring ≥2 attendees:{len(events):>8,}")

events = events[events['place_latitude'].notna()]
print(f"After requiring coordinates: {len(events):>8,}")

START_DATE, END_DATE = '2013-06-19', '2017-06-18'
date_str = events['start'].astype(str).str[:10]
events = events[(date_str >= START_DATE) & (date_str <= END_DATE)]
print(f"After date range filter:     {len(events):>8,}")

In [ ]:
# Geographic filter: keep only events within Copenhagen and Frederiksberg municipalities
gdf = gpd.GeoDataFrame(
    events,
    geometry=gpd.points_from_xy(events['place_longitude'], events['place_latitude']),
    crs='EPSG:4326'
)

dk_shape = gpd.read_file(RAW_DIR / 'dk_shape_cleaned.geojson')

gdf_dk = gpd.sjoin(gdf, dk_shape, how='inner', predicate='within')
print(f"Events in Denmark:                    {len(gdf_dk):>7,}")

target_kommunes  = ['Frederiksberg', 'København']
cph_geometry     = dk_shape.loc[dk_shape['kommune'].isin(target_kommunes), 'geometry'].union_all()
events           = gdf_dk[gdf_dk.within(cph_geometry)].copy()
print(f"Events in Copenhagen + Frederiksberg: {len(events):>7,}")

# Drop geometry and spatial join columns; add placeholders for political-attendee counts
drop_cols = [
    'end', 'updated', 'n_declined', 'declined', 'n_maybe', 'maybe',
    'n_interested', 'interested', 'n_noreply', 'canceled', 'event_category',
    'place_name', 'place_city', 'place_country', 'place_state',
    'place_latitude', 'place_longitude', 'place_street', 'place_zip',
    'noreply', 'geometry', 'index_right', 'kommune'
]
events = events.drop(columns=[c for c in drop_cols if c in events.columns])

events['n_political_attending'] = 0
events['political_attendees']   = [set() for _ in range(len(events))]
events['percent_political']     = 0.0

## 3. Build User Dataset

In [ ]:
# Collect every unique attendee ID across all events
all_attendee_ids = {
    uid
    for attendees in events['attending']
    for uid in attendees
}
print(f"Unique attendees across all events: {len(all_attendee_ids):,}")

# Intersect with political users who appear in our events
political_in_events = political_users[political_users['hashed_id'].isin(all_attendee_ids)]
print(f"Political users present in events:  {len(political_in_events):,}")

# Build the full user DataFrame (all attendees, political metadata where available)
users = pd.DataFrame({'hashed_id': list(all_attendee_ids)})
users = users.merge(political_in_events, on='hashed_id', how='left')

# Count how many events each user attended
event_counts = Counter(
    uid
    for attendees in events['attending']
    for uid in attendees
)
users['events_attended'] = users['hashed_id'].map(event_counts).fillna(0).astype(int)

# Mark users as politically active if they have an ideology score
users['political'] = users['party_lr_score'].notna()

print(f"Total users:     {len(users):,}")
print(f"Political users: {users['political'].sum():,}")

## 4. Bot Removal

Users in the top 0.01% by events attended are removed as likely bots.

In [ ]:
bot_threshold = np.percentile(users['events_attended'], 99.99)
print(f"99.99th percentile threshold: {bot_threshold:.0f} events")

n_before = len(users)
users    = users[users['events_attended'] <= bot_threshold].copy()
print(f"Removed {n_before - len(users):,} bot accounts — {len(users):,} users remain")

## 5. Iterative Cleaning

Events with no politically active attendees and users with no remaining events are removed in alternating passes until the dataset stabilises.

In [ ]:
def update_counts(events_df, users_df):
    """Recompute attendee counts for each event given the current user set."""
    valid_users    = set(users_df['hashed_id'])
    political_ids  = set(users_df.loc[users_df['political'], 'hashed_id'])

    def process(attendees):
        filtered  = attendees & valid_users
        political = filtered & political_ids
        n         = len(filtered)
        return filtered, n, len(political), political, (len(political) / n if n > 0 else 0.0)

    results = events_df['attending'].apply(process)
    events_df = events_df.copy()
    events_df['attending']            = results.apply(lambda x: x[0])
    events_df['n_attending']          = results.apply(lambda x: x[1])
    events_df['n_political_attending']= results.apply(lambda x: x[2])
    events_df['political_attendees']  = results.apply(lambda x: x[3])
    events_df['percent_political']    = results.apply(lambda x: x[4])
    return events_df


def update_event_counts_for_users(events_df, users_df):
    """Recompute events_attended for each user given the current event set."""
    valid_ids   = set(users_df['hashed_id'])
    user_counts = Counter(
        uid
        for attendees in events_df['attending']
        for uid in attendees
        if uid in valid_ids
    )
    users_df = users_df.copy()
    users_df['events_attended'] = users_df['hashed_id'].map(user_counts).fillna(0).astype(int)
    return users_df


def iterative_drop(events_df, users_df, min_political=1, drop_users=True, max_iterations=10):
    """Remove events and users below thresholds until the dataset stabilises.

    Args:
        min_political: minimum number of political attendees required per event.
        drop_users:    if True, also remove users with events_attended < 1.
    """
    for i in range(max_iterations):
        prev_events = len(events_df)
        prev_users  = len(users_df)

        if drop_users:
            users_df  = users_df[users_df['events_attended'] >= 1].copy()

        events_df = update_counts(events_df, users_df)
        users_df  = update_event_counts_for_users(events_df, users_df)

        events_df = events_df[events_df['n_political_attending'] >= min_political].copy()
        events_df = update_counts(events_df, users_df)
        users_df  = update_event_counts_for_users(events_df, users_df)

        dropped_e = prev_events - len(events_df)
        dropped_u = prev_users  - len(users_df)

        if dropped_e == 0 and dropped_u == 0:
            print(f"Converged in {i + 1} iteration(s)")
            break
        print(f"Iteration {i + 1}: -{dropped_e} events, -{dropped_u} users")

    return events_df, users_df

In [ ]:
# Compute initial counts, then run global iterative cleaning
events = update_counts(events, users)
users  = update_event_counts_for_users(events, users)

print(f"Before global cleaning: {len(events):,} events, {len(users):,} users")
events, users = iterative_drop(events, users, min_political=1, drop_users=True)
print(f"After global cleaning:  {len(events):,} events, {len(users):,} users")

## 6. Ideology Scoring

User ideology scores are normalised using a custom min-max transform centred on the midpoint of the Danish party spectrum (5.78). Scores below centre map to [−1, 0] (left); scores above map to [0, +1] (right).

Each event is then assigned a weighted ideology score:

$$\text{weighted ideology} = \overline{\text{ideology of political attendees}} \times \text{fraction politically active}$$

In [ ]:
CENTER    = 5.78
LEFT_MIN  = 1.74
RIGHT_MAX = 7.40

def normalize_ideology(x):
    if pd.isna(x):
        return np.nan
    if x < CENTER:
        return (x - CENTER) / (CENTER - LEFT_MIN)
    if x > CENTER:
        return (x - CENTER) / (RIGHT_MAX - CENTER)
    return 0.0

users['party_lr_score']    = users['party_lr_score'].round(2)
users['normalized_min_max'] = users['party_lr_score'].apply(normalize_ideology).round(2)

users['l_r_min_max'] = np.select(
    condlist=[
        users['normalized_min_max'].isna(),
        users['normalized_min_max'] > 0,
        users['normalized_min_max'] <= 0,
    ],
    choicelist=['Non-political', 'R', 'L'],
    default='Non-political'
)

print(users['l_r_min_max'].value_counts())

In [ ]:
user_ideology = users.set_index('hashed_id')['normalized_min_max'].to_dict()

def mean_ideology(political_attendees):
    scores = [user_ideology.get(uid, np.nan) for uid in political_attendees]
    valid  = [s for s in scores if pd.notna(s)]
    return round(np.mean(valid), 2) if valid else np.nan

events['average_normalized_min_max'] = events['political_attendees'].apply(mean_ideology)
events['weighted_ideology_min_max']  = (
    events['average_normalized_min_max'] * events['percent_political']
).round(2)

print(f"Events with ideology score: {events['weighted_ideology_min_max'].notna().sum():,}")
print(f"Average weighted ideology:  {events['weighted_ideology_min_max'].mean():.3f}")

## 7. Split by Event Size and Time Period

Events are split into two size categories and four annual periods running from June 19 to June 18, aligned with the Danish election calendar. Large events (> 100 attendees) are excluded from analysis.

In [ ]:
events['date'] = events['start'].astype(str).str[:10]

events_small  = events[events['n_attending'] <= 30].copy()
events_medium = events[(events['n_attending'] > 30) & (events['n_attending'] <= 100)].copy()

print(f"Small events  (≤ 30):    {len(events_small):,}")
print(f"Medium events (31–100):  {len(events_medium):,}")

In [ ]:
TIME_PERIODS = {
    '2013-2014': ('2013-06-19', '2014-06-18'),
    '2014-2015': ('2014-06-19', '2015-06-18'),
    '2015-2016': ('2015-06-19', '2016-06-18'),
    '2016-2017': ('2016-06-19', '2017-06-18'),
}

def split_by_period(events_df):
    return {
        name: events_df[
            (events_df['date'] >= start) & (events_df['date'] <= end)
        ].copy()
        for name, (start, end) in TIME_PERIODS.items()
    }

def users_for_events(events_df, users_df):
    """Return the subset of users_df who attended at least one event in events_df."""
    user_ids = {uid for attendees in events_df['attending'] for uid in attendees}
    return users_df[users_df['hashed_id'].isin(user_ids)].copy()


# Build all eight period × size datasets
datasets = {}
for size_label, size_events in [('small', events_small), ('medium', events_medium)]:
    for period_name, period_events in split_by_period(size_events).items():
        key = f'{size_label}_{period_name}'
        datasets[key] = {
            'events': period_events,
            'users' : users_for_events(period_events, users),
        }
        print(f"{key}: {len(period_events):,} events, {len(datasets[key]['users']):,} users")

## 8. Per-Period Iterative Cleaning

After splitting, each dataset is cleaned independently to remove events that have no political attendees within the specific period and size subset.

In [ ]:
for key, data in datasets.items():
    evts = update_counts(data['events'], data['users'])
    usrs = update_event_counts_for_users(evts, data['users'])

    print(f"\n{key}")
    evts, usrs = iterative_drop(evts, usrs, min_political=1, drop_users=False)
    print(f"  Final: {len(evts):,} events, {len(usrs):,} users")

    datasets[key] = {'events': evts, 'users': usrs}

## 9. Save Outputs

In [ ]:
for key, data in datasets.items():
    data['events'].to_pickle(DATA_DIR / f'events_{key}.pkl')
    data['users'].to_pickle(DATA_DIR / f'users_{key}.pkl')

print(f"Saved 16 datasets to {DATA_DIR}")
print("\nSummary:")
print(f"{'Dataset':<25} {'Events':>8} {'Users':>10} {'Political users':>16}")
print("-" * 62)
for key, data in sorted(datasets.items()):
    n_political = data['users']['political'].sum()
    print(f"{key:<25} {len(data['events']):>8,} {len(data['users']):>10,} {n_political:>16,}")